In [ ]:
# imports
from mailersend import MailerSendClient, EmailBuilder
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import gradio as gr

In [23]:
# The usual start

load_dotenv(override=True)
openai = OpenAI()

In [24]:
# For mail sender

mailer_send_api_token = os.getenv("MAILER_SEND_API_TOKEN")
mailer_send_domain = os.getenv("MAILER_SEND_DOMAIN")

if mailer_send_api_token:
    print(f"Mailer Send API Token found and starts with {mailer_send_api_token[0:5]}")
else:
    print("Mailer Send API Token not found")

if mailer_send_domain:
    print(f"Mailer Send Domain found and starts with {mailer_send_domain[0:5]}")
else:
    print("Mailer Send Domain not found")


Mailer Send API Token found and starts with mlsn.
Mailer Send Domain found and starts with test-


In [25]:
# For your data
your_email = os.getenv("YOUR_EMAIL")
your_name = os.getenv("YOUR_NAME")

if your_email:
    print(f"Your Email found and starts with {your_email[0:5]}")
else:
    print("Your Email not found")

if your_name:
    print(f"Your Name found and starts with {your_name[0:5]}")
else:
    print("Your Name not found")


Your Email found and starts with b5923
Your Name found and starts with Aphir


In [26]:
ms = MailerSendClient(api_key=mailer_send_api_token)

def send_email(sub, message, to_email):
    email = (EmailBuilder()
         .from_email(f"no-reply@{mailer_send_domain}", "No Reply")
         .to_many([{"email": to_email }])
         .subject(sub)
         .text(message)
         .build())

    response = ms.emails.send(email)
    print(f"Email sent: {response.id}")

In [19]:
send_email("Hello", "World", your_email)

Email sent: 6a313e0be4b05ff18b51ac51


In [27]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    send_email("New recording interest", f"Recording interest from {name} with email {email} and notes {notes}", your_email)
    return {"recorded": "ok"}

In [28]:
def record_unknown_question(question):
    send_email("New unknown question", f"Recording {question} asked that I couldn't answer", your_email)
    return {"recorded": "ok"}

In [30]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [31]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [32]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [33]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional information about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['quest

In [34]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [35]:
globals()["record_unknown_question"]("this is a really hard question")

Email sent: 6a313fca1be106329ca1348b


{'recorded': 'ok'}

In [39]:
with open("me/my-resume.md", "r", encoding="utf-8") as f:
    resume = f.read()

name = your_name

In [40]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Resume:\n{resume}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [41]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:

        # This is the call to the LLM - see that we pass in the tools json

        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason
        
        # If the LLM wants to call a tool, we do that!
         
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [42]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool called: record_unknown_question
Email sent: 6a31410c33b1564a173b259a
Tool called: record_unknown_question
Email sent: 6a31413f27c4a8a5e0392053
Tool called: record_unknown_question
Email sent: 6a3141907f2bd3f702654de5
Tool called: record_user_details
Email sent: 6a3141bf92aac413355fc20e
